# Traveling Salesman Problem (TSP)
## Comparing Heuristic and Exact Optimization Methods

### Techniques Used
- Google OR-Tools (PATH_CHEAPEST_ARC + Guided Local Search)
- Gurobi MILP (MTZ Formulation)
- Warm Start using OR-Tools
- K-Nearest Neighbour Sparsification

### Dataset
- 1000 Indian Cities
- Road Network Distance Matrix (OSRM)

### Objective
Compare heuristic and exact optimization approaches on increasing TSP instances and study scalability, warm-starting, and sparsification.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import folium
import time

import gurobipy as gp
from gurobipy import GRB

from ortools.constraint_solver import pywrapcp
from ortools.constraint_solver import routing_enums_pb2

### OR-Tools Solver

This function constructs and solves the TSP using Google OR-Tools.

Configuration:
- PATH_CHEAPEST_ARC
- GUIDED_LOCAL_SEARCH
- Time Limit = 30 seconds

In [10]:
def solve_tsp_ortools(distance_matrix, N):

    city_names = distance_matrix.index[:N]

    D = distance_matrix.loc[
        city_names,
        city_names
    ].values

    D = (D * 1000).astype(int)

    manager = pywrapcp.RoutingIndexManager(
        N,
        1,
        0
    )

    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):

        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)

        return int(
            D[from_node][to_node]
        )

    transit_callback_index = routing.RegisterTransitCallback(
        distance_callback
    )

    routing.SetArcCostEvaluatorOfAllVehicles(
        transit_callback_index
    )

    search_parameters = (
        pywrapcp.DefaultRoutingSearchParameters()
    )

    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )

    search_parameters.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )

    search_parameters.time_limit.seconds = 30

    start = time.time()

    solution = routing.SolveWithParameters(
        search_parameters
    )

    runtime = time.time() - start

    if solution:

        route = []

        index = routing.Start(0)

        total_distance = 0

        while not routing.IsEnd(index):

            node = manager.IndexToNode(index)

            route.append(node)

            next_index = solution.Value(
                routing.NextVar(index)
            )

            total_distance += routing.GetArcCostForVehicle(
                index,
                next_index,
                0
            )

            index = next_index

        route.append(0)

        return {
            "N": N,
            "Distance_km": total_distance / 1000,
            "Runtime_sec": runtime,
            "Route": route,
            "Status": "Solved"
        }

    return {
        "N": N,
        "Distance_km": None,
        "Runtime_sec": runtime,
        "Route": None,
        "Status": "No Solution"
    }

# Load Dataset

In [11]:
distance_matrix = pd.read_csv(r"C:\Users\DELL\Documents\TSP-Optimization-using-Gurobi-and-OR-Tools\data\distance_matrix.csv",
    
)
distance_matrix = distance_matrix.pivot(
    index="fromplace",
    columns="toplace",
    values="dist_km"
)

In [12]:

res_10 = solve_tsp_ortools(
    distance_matrix,
    10
)

print(res_10)

{'N': 10, 'Distance_km': 8370.0, 'Runtime_sec': 30.025543689727783, 'Route': [0, 6, 5, 2, 3, 4, 7, 9, 8, 1, 0], 'Status': 'Solved'}


In [13]:
instances = [10,50,100,250,500,1000]

results_task1 = []

for N in instances:

    print(f"\nRunning N={N}")

    res = solve_tsp_ortools(
        distance_matrix,
        N
    )

    print(
        f"Distance={res['Distance_km']:.2f} km"
    )

    print(
        f"Runtime={res['Runtime_sec']:.2f} sec"
    )

    results_task1.append(res)


Running N=10
Distance=8370.00 km
Runtime=30.00 sec

Running N=50
Distance=14408.00 km
Runtime=30.00 sec

Running N=100
Distance=19128.00 km
Runtime=30.00 sec

Running N=250
Distance=26883.00 km
Runtime=30.00 sec

Running N=500
Distance=37015.00 km
Runtime=30.00 sec

Running N=1000
Distance=52546.00 km
Runtime=30.02 sec


## Task 1: OR-Tools Heuristic Results

Google OR-Tools was used to generate high-quality feasible tours within a fixed time limit of 30 seconds for each instance.

The heuristic consistently produced good-quality solutions with very low computational time, making it suitable for generating initial solutions for larger optimization models.

In [14]:
task1_df = pd.DataFrame(results_task1)

task1_df

,N,Distance_km,Runtime_sec,Route,Status
0,10,8370.0,30.000092,"[0, 6, 5, 2, 3, 4, 7, 9, 8, 1, 0]",Solved
1,50,14408.0,30.001317,"[0, 19, 11, 1, 40, 30, 8, 36, 41, 9, 25, 7, 20...",Solved
2,100,19128.0,30.000231,"[0, 66, 19, 11, 77, 1, 40, 30, 8, 36, 41, 97, ...",Solved
3,250,26883.0,30.000679,"[0, 230, 113, 104, 31, 112, 38, 89, 192, 23, 2...",Solved
4,500,37015.0,30.000493,"[0, 288, 290, 283, 113, 230, 414, 199, 387, 23...",Solved
5,1000,52546.0,30.019090,"[0, 565, 612, 507, 509, 283, 290, 600, 382, 53...",Solved


In [15]:
task1_df.to_csv(
    "Task1_ORTools.csv",
    index=False
)

# Warm Start using OR-Tools Solution

A warm start provides the optimization solver with an initial feasible solution before the branch-and-bound process begins.

Instead of starting from scratch, the high-quality tour generated by Google OR-Tools is supplied as a **MIP Start** to Gurobi.

### Purpose
- Improve the quality of the initial incumbent solution.
- Reduce the optimality gap within the same time limit.
- Accelerate convergence for larger TSP instances.

The OR-Tools route is converted into binary decision variables (`xᵢⱼ`) and supplied to Gurobi through the `Start` attribute. This allows the solver to focus on improving an already feasible solution rather than searching for one from scratch.

The performance of the warm-start model is compared with the cold-start MILP model using runtime, objective value, and MIP gap.

In [13]:
def solve_tsp_gurobi_warm(
    distance_matrix,
    N,
    route
):

    import gurobipy as gp
    from gurobipy import GRB
    import time

    city_names = distance_matrix.index[:N]

    D = distance_matrix.loc[
        city_names,
        city_names
    ].values

    model = gp.Model(
        f"TSP_WARM_{N}"
    )

    model.setParam(
        "TimeLimit",
        1800
    )

    # -------------------------
    # Variables
    # -------------------------

    x = {}

    for i in range(N):
        for j in range(N):

            if i != j:

                x[i,j] = model.addVar(
                    vtype=GRB.BINARY,
                    name=f"x_{i}_{j}"
                )

    u = {}

    for i in range(1,N):

        u[i] = model.addVar(
            lb=1,
            ub=N-1,
            vtype=GRB.CONTINUOUS,
            name=f"u_{i}"
        )

    model.update()

    # -------------------------
    # Warm Start
    # -------------------------

    for i,j in x.keys():

        x[i,j].Start = 0

    for k in range(len(route)-1):

        i = route[k]
        j = route[k+1]

        if (i,j) in x:

            x[i,j].Start = 1

    for pos, city in enumerate(
        route[1:-1],
        start=1
    ):

        if city in u:

            u[city].Start = pos

    # -------------------------
    # Objective
    # -------------------------

    model.setObjective(

        gp.quicksum(
            D[i,j] * x[i,j]
            for i in range(N)
            for j in range(N)
            if i != j
        ),

        GRB.MINIMIZE

    )

    # -------------------------
    # Outgoing
    # -------------------------

    for i in range(N):

        model.addConstr(

            gp.quicksum(
                x[i,j]
                for j in range(N)
                if i != j
            ) == 1

        )

    # -------------------------
    # Incoming
    # -------------------------

    for j in range(N):

        model.addConstr(

            gp.quicksum(
                x[i,j]
                for i in range(N)
                if i != j
            ) == 1

        )

    # -------------------------
    # MTZ
    # -------------------------

    for i in range(1,N):
        for j in range(1,N):

            if i != j:

                model.addConstr(

                    u[i]
                    - u[j]
                    + N*x[i,j]
                    <= N-1

                )

    # -------------------------
    # Solve
    # -------------------------

    
    print("Starting optimization...")
    start = time.time()

    

    end = time.time()

    print("Solver runtime:", end-start)
    print("Gurobi runtime:", model.Runtime)

    model.optimize()

    runtime = model.Runtime

    result = {

        "N": N,
        "ObjVal": None,
        "ObjBound": None,
        "Gap": None,
        "Runtime_sec": runtime,
        "Status": model.Status

    }

    if model.SolCount > 0:

        result["ObjVal"] = model.ObjVal
        result["ObjBound"] = model.ObjBound
        result["Gap"] = model.MIPGap * 100

    return result

In [14]:
route_10 = results_task1[0]["Route"]

warm_10 = solve_tsp_gurobi_warm(
    distance_matrix,
    10,
    route_10
)

print(warm_10)

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2835067
Academic license 2835067 - for non-commercial use only - registered to mq___@isibang.ac.in
Set parameter TimeLimit to value 1800
Starting optimization...
Solver runtime: 2.384185791015625e-07
Gurobi runtime: 0.0
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 5 7520U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  1800

Academic license 2835067 - for non-commercial use only - registered to mq___@isibang.ac.in
Optimize a model with 92 rows, 99 columns and 396 nonzeros (Min)
Model fingerprint: 0x5f49b726
Model has 90 linear objective coefficients
Variable types: 9 continuous, 90 integer (90 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+01]
  Objective range  [2e+02, 3e+03]
  Bounds range     [1e+00, 9e+00]
  RH

# Exact Solution using Gurobi (MILP)

In this section, we formulate the Traveling Salesman Problem (TSP) as a Mixed Integer Linear Programming (MILP) model using the Miller-Tucker-Zemlin (MTZ) formulation.

### Model Components
- **Decision Variables (`xᵢⱼ`)**: Binary variables indicating whether the route travels directly from city *i* to city *j*.
- **MTZ Variables (`uᵢ`)**: Continuous variables used to eliminate subtours.
- **Objective**: Minimize the total travel distance.
- **Constraints**:
  - Each city has exactly one outgoing edge.
  - Each city has exactly one incoming edge.
  - MTZ constraints prevent the formation of subtours.

The model is solved using the Gurobi Optimizer with a fixed time limit. For larger problem instances, the solver may terminate before proving optimality, in which case the best feasible solution and optimality gap are reported.

In [16]:
def solve_tsp_gurobi(
    distance_matrix,
    N
):

    city_names = distance_matrix.index[:N]

    D = distance_matrix.loc[
        city_names,
        city_names
    ].values

    model = gp.Model(
        f"TSP_{N}"
    )

    model.setParam(
        "TimeLimit",
        1800
    )

    x = {}

    for i in range(N):
        for j in range(N):

            if i != j:

                x[i,j] = model.addVar(
                    vtype=GRB.BINARY
                )

    u = {}

    for i in range(1,N):

        u[i] = model.addVar(
            lb=1,
            ub=N-1,
            vtype=GRB.CONTINUOUS
        )

    model.update()

    model.setObjective(

        gp.quicksum(
            D[i,j] * x[i,j]
            for i in range(N)
            for j in range(N)
            if i != j
        ),

        GRB.MINIMIZE

    )

    for i in range(N):

        model.addConstr(

            gp.quicksum(
                x[i,j]
                for j in range(N)
                if i != j
            ) == 1

        )

    for j in range(N):

        model.addConstr(

            gp.quicksum(
                x[i,j]
                for i in range(N)
                if i != j
            ) == 1

        )

    for i in range(1,N):
        for j in range(1,N):

            if i != j:

                model.addConstr(

                    u[i]
                    - u[j]
                    + N*x[i,j]
                    <= N-1

                )

    start = time.time()

    model.optimize()

    runtime = runtime = model.Runtime

    result = {

        "N": N,
        "Runtime_sec": runtime,
        "Status": model.Status

    }

    if model.SolCount > 0:

     result["ObjVal"] = model.ObjVal
     result["ObjBound"] = model.ObjBound
     result["Gap"] = model.MIPGap * 100

    return result

In [17]:
route_10 = results_task1[0]["Route"]

cold_10 = solve_tsp_gurobi(
    distance_matrix,
    10
)

warm_10 = solve_tsp_gurobi_warm(
    distance_matrix,
    10,
    route_10
)

print(cold_10)
print(warm_10)

Set parameter TimeLimit to value 1800
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 5 7520U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  1800

Academic license 2835067 - for non-commercial use only - registered to mq___@isibang.ac.in
Optimize a model with 92 rows, 99 columns and 396 nonzeros (Min)
Model fingerprint: 0x95a47b37
Model has 90 linear objective coefficients
Variable types: 9 continuous, 90 integer (90 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+01]
  Objective range  [2e+02, 3e+03]
  Bounds range     [1e+00, 9e+00]
  RHS range        [1e+00, 9e+00]

Found heuristic solution: objective 12872.000000
Presolve time: 0.00s
Presolved: 92 rows, 99 columns, 795 nonzeros
Variable types: 9 continuous, 90 integer (90 binary)

Root relaxation: objective 7.948162e+03, 26 iterations, 0.00 seco

In [18]:
comparison_10 = pd.DataFrame(
    [
        {
            "Mode":"Cold",
            **cold_10
        },
        {
            "Mode":"Warm",
            **warm_10
        }
    ]
)

comparison_10

,Mode,N,Runtime_sec,Status,ObjVal,ObjBound,Gap
0,Cold,10,0.084,2,8370.0,8370.0,0.0
1,Warm,10,0.092,2,8370.0,8370.0,0.0


In [19]:
route_50 = results_task1[1]["Route"]

warm_50 = solve_tsp_gurobi_warm(
    distance_matrix,
    50,
    route_50
)

print(warm_50)

Set parameter TimeLimit to value 1800
Starting optimization...
Solver runtime: 2.384185791015625e-07
Gurobi runtime: 0.0
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 5 7520U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  1800

Academic license 2835067 - for non-commercial use only - registered to mq___@isibang.ac.in
Optimize a model with 2452 rows, 2499 columns and 11956 nonzeros (Min)
Model fingerprint: 0x8cb0af69
Model has 2450 linear objective coefficients
Variable types: 49 continuous, 2450 integer (2450 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+01]
  Objective range  [8e+00, 4e+03]
  Bounds range     [1e+00, 5e+01]
  RHS range        [1e+00, 5e+01]

Loaded user MIP start with objective 14408

Presolve time: 0.02s
Presolved: 2452 rows, 2499 columns, 11956 nonzeros
Variable types: 49 con

In [20]:
cold_50 = solve_tsp_gurobi(
    distance_matrix,
    50
)

print(cold_50)

Set parameter TimeLimit to value 1800
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 5 7520U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  1800

Academic license 2835067 - for non-commercial use only - registered to mq___@isibang.ac.in
Optimize a model with 2452 rows, 2499 columns and 11956 nonzeros (Min)
Model fingerprint: 0x7c540698
Model has 2450 linear objective coefficients
Variable types: 49 continuous, 2450 integer (2450 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+01]
  Objective range  [8e+00, 4e+03]
  Bounds range     [1e+00, 5e+01]
  RHS range        [1e+00, 5e+01]

Presolve time: 0.04s
Presolved: 2452 rows, 2499 columns, 11956 nonzeros
Variable types: 49 continuous, 2450 integer (2450 binary)
Found heuristic solution: objective 66555.000000
Found heuristic solution: objective 61764.

In [21]:
comparison_50 = pd.DataFrame(
    [
        {
            "Mode":"Cold",
            **cold_50
        },
        {
            "Mode":"Warm",
            **warm_50
        }
    ]
)

comparison_50

,Mode,N,Runtime_sec,Status,ObjVal,ObjBound,Gap
0,Cold,50,1800.196,9,14425.0,14196.0,1.587522
1,Warm,50,1800.115,9,14408.0,14253.0,1.075791


In [22]:
print(cold_50)
print(warm_50)

{'N': 50, 'Runtime_sec': 1800.1960000991821, 'Status': 9, 'ObjVal': 14425.0, 'ObjBound': 14196.0, 'Gap': 1.587521663778163}
{'N': 50, 'ObjVal': 14408.0, 'ObjBound': 14253.0, 'Gap': 1.075791227096058, 'Runtime_sec': 1800.1150000095367, 'Status': 9}


In [23]:
route_100 = results_task1[2]["Route"]

cold_100 = solve_tsp_gurobi(
    distance_matrix,
    100
)

warm_100 = solve_tsp_gurobi_warm(
    distance_matrix,
    100,
    route_100
)

print(cold_100)
print(warm_100)

Set parameter TimeLimit to value 1800
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 5 7520U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  1800

Academic license 2835067 - for non-commercial use only - registered to mq___@isibang.ac.in
Optimize a model with 9902 rows, 9999 columns and 48906 nonzeros (Min)
Model fingerprint: 0xaee3978c
Model has 9900 linear objective coefficients
Variable types: 99 continuous, 9900 integer (9900 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+02]
  Objective range  [2e+00, 4e+03]
  Bounds range     [1e+00, 1e+02]
  RHS range        [1e+00, 1e+02]

Presolve time: 0.08s
Presolved: 9902 rows, 9999 columns, 48906 nonzeros
Variable types: 99 continuous, 9900 integer (9900 binary)

Root relaxation: objective 1.340069e+04, 407 iterations, 0.04 seconds (0.03 work units)

 

In [24]:
comparison_100 = pd.DataFrame(
    [
        {"Mode":"Cold", **cold_100},
        {"Mode":"Warm", **warm_100}
    ]
)

comparison_100

,Mode,N,Runtime_sec,Status,ObjVal,ObjBound,Gap
0,Cold,100,1800.286,9,19896.0,18376.0,7.639727
1,Warm,100,1800.086,9,18850.0,18585.0,1.405836


In [25]:
route_250 = results_task1[3]["Route"]

cold_250 = solve_tsp_gurobi(
    distance_matrix,
    250
)

warm_250 = solve_tsp_gurobi_warm(
    distance_matrix,
    250,
    route_250
)

print(cold_250)
print(warm_250)

Set parameter TimeLimit to value 1800
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 5 7520U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  1800

Academic license 2835067 - for non-commercial use only - registered to mq___@isibang.ac.in
Optimize a model with 62252 rows, 62499 columns and 309756 nonzeros (Min)
Model fingerprint: 0xfb97034f
Model has 62250 linear objective coefficients
Variable types: 249 continuous, 62250 integer (62250 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+02]
  Objective range  [1e+00, 4e+03]
  Bounds range     [1e+00, 2e+02]
  RHS range        [1e+00, 2e+02]

Presolve time: 0.75s
Presolved: 62252 rows, 62499 columns, 309756 nonzeros
Variable types: 249 continuous, 62250 integer (62250 binary)
Deterministic concurrent LP optimizer: primal and dual simplex
Showing primal 

## Task 2: Warm-Start vs Cold-Start MILP

The OR-Tools routes were provided to Gurobi as MIP Start solutions.

The objective of this experiment is to evaluate whether supplying an initial feasible solution improves the branch-and-bound search compared to solving the MILP from scratch.

The comparison focuses on:
- Best feasible objective value
- Remaining optimality gap
- Runtime
- Solver status

In [26]:
task2_results = [
    {"Mode":"Cold", **cold_50},
    {"Mode":"Warm", **warm_50},
    {"Mode":"Cold", **cold_100},
    {"Mode":"Warm", **warm_100},
    {"Mode":"Cold", **cold_250},
    {"Mode":"Warm", **warm_250},
]

task2_df = pd.DataFrame(task2_results)

task2_df

,Mode,N,Runtime_sec,Status,ObjVal,ObjBound,Gap
0,Cold,50,1800.196,9,14425.0,14196.0,1.587522
1,Warm,50,1800.115,9,14408.0,14253.0,1.075791
2,Cold,100,1800.286,9,19896.0,18376.0,7.639727
3,Warm,100,1800.086,9,18850.0,18585.0,1.405836
4,Cold,250,1800.602,9,29176.0,25105.0,13.953249
5,Warm,250,1800.213,9,26521.0,25321.0,4.524716


In [27]:
task2_df.to_csv(
    "Task2_WarmStart.csv",
    index=False
)

In [28]:
route_500 = results_task1[4]["Route"]

cold_500 = solve_tsp_gurobi(
    distance_matrix,
    500
)

warm_500 = solve_tsp_gurobi_warm(
    distance_matrix,
    500,
    route_500
)




Set parameter TimeLimit to value 1800


Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 5 7520U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  1800

Academic license 2835067 - for non-commercial use only - registered to mq___@isibang.ac.in
Optimize a model with 249502 rows, 249999 columns and 1244506 nonzeros (Min)
Model fingerprint: 0xb120642e
Model has 249500 linear objective coefficients
Variable types: 499 continuous, 249500 integer (249500 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+02]
  Objective range  [1e+00, 4e+03]
  Bounds range     [1e+00, 5e+02]
  RHS range        [1e+00, 5e+02]

Presolve time: 3.06s
Presolved: 249502 rows, 249999 columns, 1244506 nonzeros
Variable types: 499 continuous, 249500 integer (249500 binary)
Deterministic concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only

In [29]:
print(cold_500)
print(warm_500)

{'N': 500, 'Runtime_sec': 1802.970999956131, 'Status': 9, 'ObjVal': 41895.0, 'ObjBound': 33409.0, 'Gap': 20.255400405776346}
{'N': 500, 'ObjVal': 36331.0, 'ObjBound': 33251.0, 'Gap': 8.477608653766755, 'Runtime_sec': 1801.0409998893738, 'Status': 9}


In [30]:
comparison_500 = pd.DataFrame(
    [
        {"Mode":"Cold", **cold_500},
        {"Mode":"Warm", **warm_500}
    ]
)

comparison_500

,Mode,N,Runtime_sec,Status,ObjVal,ObjBound,Gap
0,Cold,500,1802.971,9,41895.0,33409.0,20.255400
1,Warm,500,1801.041,9,36331.0,33251.0,8.477609


### Observations

- Warm-start consistently produced better incumbent solutions than the cold-start model.
- The MIP gap decreased significantly after providing an initial feasible solution.
- The improvement became more noticeable for larger problem instances.
- Both approaches reached the specified time limit for larger instances, but warm-starting produced higher-quality solutions within the same computational budget.

In [31]:
route_1000 = results_task1[5]["Route"]

cold_1000 = solve_tsp_gurobi(
    distance_matrix,
    1000
)

warm_1000 = solve_tsp_gurobi_warm(
    distance_matrix,
    1000,
    route_1000
)

print(cold_1000)
print(warm_1000)

Set parameter TimeLimit to value 1800
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 5 7520U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  1800

Academic license 2835067 - for non-commercial use only - registered to mq___@isibang.ac.in
Optimize a model with 999002 rows, 999999 columns and 4989006 nonzeros (Min)
Model fingerprint: 0x1ccd8ff8
Model has 999000 linear objective coefficients
Variable types: 999 continuous, 999000 integer (999000 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+03]
  Objective range  [1e+00, 4e+03]
  Bounds range     [1e+00, 1e+03]
  RHS range        [1e+00, 1e+03]

Presolve removed 0 rows and 0 columns (presolve time = 5s)...
Presolve removed 0 rows and 0 columns (presolve time = 10s)...
Presolve time: 12.16s
Presolved: 999002 rows, 999999 columns, 4989006 nonzeros
Vari

GurobiError: Out of memory

In [32]:
warm_1000 = solve_tsp_gurobi_warm(
    distance_matrix,
    1000,
    route_1000
)

print(cold_1000)
print(warm_1000)

Set parameter TimeLimit to value 1800
Starting optimization...
Solver runtime: 2.384185791015625e-07
Gurobi runtime: 0.0
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 5 7520U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  1800

Academic license 2835067 - for non-commercial use only - registered to mq___@isibang.ac.in
Optimize a model with 999002 rows, 999999 columns and 4989006 nonzeros (Min)
Model fingerprint: 0x0f4cd146
Model has 999000 linear objective coefficients
Variable types: 999 continuous, 999000 integer (999000 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+03]
  Objective range  [1e+00, 4e+03]
  Bounds range     [1e+00, 1e+03]
  RHS range        [1e+00, 1e+03]

Loaded user MIP start with objective 52374

Presolve removed 0 rows and 0 columns (presolve time = 5s)...
Presolve removed 0 

GurobiError: Out of memory

In [33]:
comparison_1000 = pd.DataFrame(
    [
        {"Mode":"Cold", **cold_1000},
        {"Mode":"Warm", **warm_1000}
    ]
)

comparison_1000

NameError: name 'cold_1000' is not defined

# K-Nearest Neighbour (KNN) Sparsification

The complete MILP model contains an edge between every pair of cities, resulting in a very large number of decision variables and constraints for large problem instances.

To improve computational efficiency, a **K-Nearest Neighbour (KNN) sparsification** strategy is adopted.

### Approach
- For each city, only the **K nearest neighbouring cities** are retained as candidate edges.
- All other edges are removed from the optimization model.
- The resulting graph is symmetrized to preserve connectivity.

### Objective
Reduce:
- Number of binary variables
- Number of constraints
- Memory consumption
- Solver runtime

Different values of **K** are evaluated to study the trade-off between computational efficiency and solution quality. As K increases, the solution quality generally improves, while the computational cost also increases.

In [34]:
import numpy as np

def create_knn_arcs(distance_matrix, N, K):

    city_names = distance_matrix.index[:N]

    D = distance_matrix.loc[
        city_names,
        city_names
    ]

    arcs = set()

    for i in range(N):

        distances = D.iloc[i].values

        nearest = np.argsort(
            distances
        )[1:K+1]

        for j in nearest:

            arcs.add((i,j))

            # Symmetrize
            arcs.add((j,i))

    return arcs

In [35]:
for K in [10,20,50]:

    arcs = create_knn_arcs(
        distance_matrix,
        500,
        K
    )

    print(
        f"K={K}, Arcs={len(arcs)}"
    )

K=10, Arcs=6146
K=20, Arcs=12316
K=50, Arcs=30862


In [36]:
def solve_tsp_knn(
    distance_matrix,
    N,
    K,
    timelimit=1800
):

    import gurobipy as gp
    from gurobipy import GRB

    city_names = distance_matrix.index[:N]

    D = distance_matrix.loc[
        city_names,
        city_names
    ].values

    arcs = create_knn_arcs(
        distance_matrix,
        N,
        K
    )

    model = gp.Model(
        f"TSP_KNN_{N}_{K}"
    )

    model.setParam(
        "TimeLimit",
        timelimit
    )

    # -----------------
    # Variables
    # -----------------

    x = {}

    for i,j in arcs:

        x[i,j] = model.addVar(
            vtype=GRB.BINARY,
            name=f"x_{i}_{j}"
        )

    u = {}

    for i in range(1,N):

        u[i] = model.addVar(
            lb=1,
            ub=N-1,
            name=f"u_{i}"
        )

    model.update()

    # -----------------
    # Objective
    # -----------------

    model.setObjective(

        gp.quicksum(
            D[i,j] * x[i,j]
            for i,j in arcs
        ),

        GRB.MINIMIZE

    )

    # -----------------
    # Outgoing
    # -----------------

    for i in range(N):

        outgoing = [
            x[i,j]
            for (a,j) in arcs
            if a == i
        ]

        if outgoing:

            model.addConstr(
                gp.quicksum(
                    outgoing
                ) == 1
            )

    # -----------------
    # Incoming
    # -----------------

    for j in range(N):

        incoming = [
            x[i,j]
            for (i,b) in arcs
            if b == j
        ]

        if incoming:

            model.addConstr(
                gp.quicksum(
                    incoming
                ) == 1
            )

    # -----------------
    # MTZ
    # -----------------

    for i,j in arcs:

        if i != 0 and j != 0:

            model.addConstr(

                u[i]
                - u[j]
                + N*x[i,j]
                <= N-1

            )

    # -----------------
    # Solve
    # -----------------

    model.optimize()

    result = {

        "N": N,
        "K": K,
        "Arcs": len(arcs),
        "Status": model.Status,
        "Runtime_sec": model.Runtime,
        "ObjVal": None,
        "ObjBound": None,
        "Gap": None

    }

    if model.SolCount > 0:

        result["ObjVal"] = model.ObjVal
        result["ObjBound"] = model.ObjBound
        result["Gap"] = model.MIPGap * 100

    return result

In [37]:
knn10 = solve_tsp_knn(
    distance_matrix,
    500,
    10,
    timelimit=900
)

print(knn10)

Set parameter TimeLimit to value 900
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 5 7520U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  900

Academic license 2835067 - for non-commercial use only - registered to mq___@isibang.ac.in
Optimize a model with 7124 rows, 6645 columns and 30664 nonzeros (Min)
Model fingerprint: 0xa8e7fbae
Model has 6146 linear objective coefficients
Variable types: 499 continuous, 6146 integer (6146 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+02]
  Objective range  [1e+00, 7e+02]
  Bounds range     [1e+00, 5e+02]
  RHS range        [1e+00, 5e+02]

Presolve time: 0.33s
Presolved: 7124 rows, 6645 columns, 61331 nonzeros
Variable types: 499 continuous, 6146 integer (6146 binary)

Root relaxation: objective 2.688721e+04, 2085 iterations, 0.09 seconds (0.03 work units)



In [38]:
results_knn = []

for K in [10,20,50]:

    print(f"\nRunning K={K}")

    res = solve_tsp_knn(
        distance_matrix,
        500,
        K,
        timelimit=900
    )

    print(res)

    results_knn.append(res)


Running K=10
Set parameter TimeLimit to value 900
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 5 7520U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  900

Academic license 2835067 - for non-commercial use only - registered to mq___@isibang.ac.in
Optimize a model with 7124 rows, 6645 columns and 30664 nonzeros (Min)
Model fingerprint: 0xa8e7fbae
Model has 6146 linear objective coefficients
Variable types: 499 continuous, 6146 integer (6146 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+02]
  Objective range  [1e+00, 7e+02]
  Bounds range     [1e+00, 5e+02]
  RHS range        [1e+00, 5e+02]

Presolve time: 0.23s
Presolved: 7124 rows, 6645 columns, 61331 nonzeros
Variable types: 499 continuous, 6146 integer (6146 binary)

Root relaxation: objective 2.688721e+04, 2085 iterations, 0.07 seconds (0.03

## Task 3: K-Nearest Neighbour Sparsification

Instead of considering every possible edge, the optimization model was restricted to the K nearest neighbours of each city.

This experiment evaluates the trade-off between:

- Computational complexity
- Memory consumption
- Solution quality

Different values of K were tested to study how graph sparsification affects optimization performance.

In [39]:
task3_df = pd.DataFrame(
    results_knn
)

task3_df

,N,K,Arcs,Status,Runtime_sec,ObjVal,ObjBound,Gap
0,500,10,6146,9,900.105,35802.0,34440.0,3.804257
1,500,20,12316,9,900.449,39125.0,34380.0,12.127796
2,500,50,30862,9,900.159,38877.0,33798.0,13.064280


### Observations

- Smaller values of K significantly reduced the number of decision variables.
- Solver runtime decreased due to the reduced search space.
- Increasing K generally improved solution quality but increased computational effort.
- KNN sparsification provides an effective balance between optimization accuracy and computational efficiency for large-scale TSP instances.

In [40]:
task3_df.to_csv(
    "Task3_KNN.csv",
    index=False
)

# Conclusions

This project compared heuristic and exact optimization techniques for solving the Traveling Salesman Problem on real-world road network data.

### Key Findings

- Google OR-Tools generated high-quality feasible tours within a short computation time, making it an excellent heuristic for large-scale instances.
- The MTZ-based MILP formulation successfully obtained optimal or near-optimal solutions for smaller problem instances but became computationally expensive as the number of cities increased.
- Using the OR-Tools solution as a warm start substantially improved the quality of the incumbent solution and reduced the optimality gap under identical time limits.
- K-Nearest Neighbour sparsification reduced the size of the optimization model and improved computational efficiency while maintaining competitive solution quality.

### Limitations

- The MTZ formulation scales poorly for very large instances because of the large number of binary variables and subtour elimination constraints.
- Large instances may exceed available memory before optimality can be proven.

### Future Work

Potential improvements include:

- Stronger subtour elimination methods (Lazy Constraints / Branch-and-Cut)
- Christofides or Lin-Kernighan heuristics
- Parallel optimization
- Metaheuristics such as Genetic Algorithms, Simulated Annealing and Ant Colony Optimization
- Vehicle Routing Problem (VRP) extensions with capacity and time-window constraints